# Set 07 – Naive Bayes im Vergleich mit logistischer Regression

Beide Modelle liefern Klassenwahrscheinlichkeiten, lernen aber unterschiedlich:

- Logistic Regression lernt direkt P(y | x) und eine lineare Trennfläche.
- Naive Bayes lernt P(x | y) sowie P(y), also klassenabhängige Merkmalsverteilungen.

Wir vergleichen beide Modelle auf identischen Trainings- und Testdaten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. Kontrollierter Datensatz

Die beiden numerischen Merkmale sind nicht vollkommen unabhängig. Damit ist die naive Annahme nur näherungsweise erfüllt.

In [ ]:
X_array, y_array = make_classification(
    n_samples=900,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.68, 0.32],
    class_sep=1.05,
    flip_y=0.06,
    random_state=12,
)
X = pd.DataFrame(X_array, columns=["merkmal_1", "merkmal_2"])
y = pd.Series(y_array, name="klasse")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## 2. Modelle definieren

In [ ]:
modelle = {
    "Gaussian Naive Bayes": GaussianNB(),
    "Logistische Regression": Pipeline([
        ("skalierung", StandardScaler()),
        ("modell", LogisticRegression()),
    ]),
}

for modell in modelle.values():
    modell.fit(X_train, y_train)

## 3. Testmetriken vergleichen

Neben Accuracy betrachten wir Precision, Recall, F1 und ROC-AUC für Klasse 1.

In [ ]:
zeilen = []
vorhersagen = {}
wahrscheinlichkeiten = {}

for name, modell in modelle.items():
    pred = modell.predict(X_test)
    proba = modell.predict_proba(X_test)[:, 1]
    vorhersagen[name] = pred
    wahrscheinlichkeiten[name] = proba
    zeilen.append({
        "Modell": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, proba),
    })

testvergleich = pd.DataFrame(zeilen).set_index("Modell")
display(testvergleich.round(3))

## 4. Konfusionsmatrizen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, pred) in zip(axes, vorhersagen.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 5. Entscheidungsflächen

Logistische Regression erzeugt eine lineare Grenze. Gaussian Naive Bayes kann durch unterschiedliche Varianzen je Klasse eine gekrümmte Grenze erzeugen.

In [ ]:
x1 = np.linspace(X["merkmal_1"].min() - 0.5, X["merkmal_1"].max() + 0.5, 240)
x2 = np.linspace(X["merkmal_2"].min() - 0.5, X["merkmal_2"].max() + 0.5, 240)
gitter_1, gitter_2 = np.meshgrid(x1, x2)
gitter = pd.DataFrame({
    "merkmal_1": gitter_1.ravel(),
    "merkmal_2": gitter_2.ravel(),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (name, modell) in zip(axes, modelle.items()):
    proba = modell.predict_proba(gitter)[:, 1].reshape(gitter_1.shape)
    flaeche = ax.contourf(
        gitter_1, gitter_2, proba,
        levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.65
    )
    ax.contour(gitter_1, gitter_2, proba, levels=[0.5], colors="black", linewidths=2)
    ax.scatter(X_test["merkmal_1"], X_test["merkmal_2"], c=y_test, cmap="bwr", edgecolor="white", s=28)
    ax.set_title(name)
    ax.set_xlabel("Merkmal 1")
    ax.set_ylabel("Merkmal 2")

fig.colorbar(flaeche, ax=axes, label="P(Klasse 1)")
plt.show()

## 6. Stabilerer Vergleich mit Cross-Validation

Ein einzelner Split kann zufällig leicht oder schwer sein. Deshalb vergleichen wir beide Modelle zusätzlich über fünf stratifizierte Folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_zeilen = []
for name, modell in modelle.items():
    ergebnis = cross_validate(modell, X, y, cv=cv, scoring=scoring)
    cv_zeilen.append({
        "Modell": name,
        "Accuracy_Mittel": ergebnis["test_accuracy"].mean(),
        "Recall_Mittel": ergebnis["test_recall"].mean(),
        "F1_Mittel": ergebnis["test_f1"].mean(),
        "ROC_AUC_Mittel": ergebnis["test_roc_auc"].mean(),
        "ROC_AUC_Std": ergebnis["test_roc_auc"].std(),
    })

cv_vergleich = pd.DataFrame(cv_zeilen).set_index("Modell")
display(cv_vergleich.round(3))

## Wann welches Modell?

Gaussian Naive Bayes:

- sehr schnell
- schätzt klassenabhängige Verteilungen
- kann mit wenig Daten gut funktionieren
- starke, oft verletzte Unabhängigkeitsannahme
- besonders verbreitet in einfacher Textklassifikation

Logistische Regression:

- lernt die Klassengrenze direkt
- benötigt keine Verteilungsannahme je Klasse
- kann mit korrelierten Merkmalen häufig besser umgehen
- liefert interpretierbare Koeffizienten
- besitzt standardmäßig Regularisierung

Die Entscheidung sollte über Cross-Validation, passende Metriken und fachliche Fehlerkosten getroffen werden.